install packages if you have not already installed 

In [9]:
import os
import json
import time
import hashlib
import re

from typing import List, Dict, Any, Optional
from datetime import datetime, timedelta
from dataclasses import dataclass
from collections import defaultdict

import logging

import numpy as np
from pydantic import BaseModel, Field

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage

from dotenv import load_dotenv

load_dotenv()

embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

build the cache layer

In [13]:
@dataclass
class CacheEntry:
    query: str
    response: str
    embedding: np.ndarray
    timestamp: datetime
    hits: int = 0
    cost_saved: float = 0.0


class SemanticCache:
    def __init__(self, similarity_threshold: float = 0.95, ttl_hours: int = 24):
        self.exact_cache: Dict[str, CacheEntry] = {}
        self.semantic_cache: List[CacheEntry] = []

        self.similarity_threshold = similarity_threshold
        self.ttl = timedelta(hours=ttl_hours)

        self.stats = {
            "exact_hits": 0,
            "semantic_hits": 0,
            "misses": 0,
            "total_cost_saved": 0.0
        }

    def _hash_query(self, query: str) -> str:
        return hashlib.sha256(query.lower().strip().encode()).hexdigest()

    def _is_expired(self, entry: CacheEntry) -> bool:
        return datetime.now() - entry.timestamp > self.ttl

    def _get_embedding(self, text: str):
        return np.array(embedding.embed_query(text))

    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

    def get(self, query: str, estimated_cost: float = 0.05) -> Optional[str]:
        query_hash = self._hash_query(query)

        # Exact cache
        if query_hash in self.exact_cache:
            entry = self.exact_cache[query_hash]

            if not self._is_expired(entry):
                entry.hits += 1
                entry.cost_saved += estimated_cost
                self.stats["exact_hits"] += 1
                self.stats["total_cost_saved"] += estimated_cost

                print("HIT - EXACT")
                return entry.response

        # Similarity cache
        query_embedding = self._get_embedding(query)

        for entry in self.semantic_cache:
            if self._is_expired(entry):
              continue

            similarity = self._cosine_similarity(
              query_embedding, entry.embedding
            )

            if similarity >= self.similarity_threshold:
                entry.hits += 1
                entry.cost_saved += estimated_cost
                self.stats["semantic_hits"] += 1
                self.stats["total_cost_saved"] += estimated_cost

                print("HIT - SIMILARITY")
            
                return entry.response

        self.stats["misses"]+= 1
        print("MISS")
        return None

    def set(self, query: str, response: str):
        query_hash = self._hash_query(query)
        embedding = self._get_embedding(query)

        entry = CacheEntry(
            query=query,
            response=response,
            embedding=embedding,
            timestamp=datetime.now()
        )

        self.exact_cache[query_hash] = entry
        self.semantic_cache.append(entry)


        
    def get_stats(self) -> Dict[str, Any]:
        total_queries = sum([
            self.stats["exact_hits"],
            self.stats["semantic_hits"],
            self.stats["misses"]
        ])

        hit_rate = (
            self.stats["exact_hits"] + self.stats["semantic_hits"]
        ) / total_queries if total_queries > 0 else 0

        return {
            **self.stats,
            "total_queries": total_queries,
            "hit_rate": hit_rate,
            "cache_size": len(self.exact_cache)
        }

In [14]:
cache = SemanticCache(similarity_threshold = 0.95,ttl_hours=24)

In [15]:
result1 = cache.get("where is my order #12345?")
print(result1)

MISS
None


In [18]:
cache.set("where is my order #12345?","Your order is in transit...")

In [19]:
result2 = cache.get("where is my order #12345?")
print(result2)

HIT - EXACT
Your order is in transit...


In [20]:
result3 = cache.get("what is the status of my order #12345?")
print(result3)

HIT - SIMILARITY
Your order is in transit...


observability and metrics
#how much time each query consume
#what are the tools agent have used 
#what is the cost

In [21]:
#Agent matrics 

class AgentMetrics:
    def __init__(self):
        self.query_log: List[Dict[str, Any]] = []
        self.tool_latencies = defaultdict(list)
        self.tool_errors = defaultdict(int)

    def log_query(self, query_data: Dict[str, Any]):
        """Log a complete query execution."""
        self.query_log.append({
            **query_data,
            "timestamp": datetime.now().isoformat()
        })

    def log_tool_call(self, tool_name: str, latency: float, success: bool):
        """Log individual tool execution."""
        self.tool_latencies[tool_name].append(latency)

        if not success:
            self.tool_errors[tool_name] += 1

    def get_summary(self) -> Dict[str, Any]:
        """Get aggregated metrics."""
        if not self.query_log:
           return {"message": "No queries logged yet"}

        total_queries = len(self.query_log)
        avg_latency = np.mean([q.get("total_latency", 0) for q in self.query_log])
        avg_cost = np.mean([q.get("estimated_cost", 0) for q in self.query_log])
        avg_steps = np.mean([q.get("steps", 0) for q in self.query_log])

        # Tool stats
        tool_stats = {}

        for tool, latencies in self.tool_latencies.items():
            tool_stats[tool] = {
                "calls": len(latencies),
                "avg_latency_ms": np.mean(latencies) * 1000,
                "p95_latency_ms": (
                    np.percentile(latencies, 95) * 1000
                    if len(latencies) > 1
                    else latencies[0] * 1000
                ),
                "errors": self.tool_errors.get(tool, 0)
            }

        return {
            "total_queries": total_queries,
            "avg_latency_sec": round(avg_latency, 3),
            "avg_cost_usd": round(avg_cost, 4),
            "avg_steps": round(avg_steps, 2),
            "tool_stats": tool_stats
        }

    def print_trace(self, query_index: int = -1):
        """Pretty print execution trace for a query."""
        if not self.query_log:
            print("No queries to trace")
            return

        query = self.query_log[query_index]

        print("\n" + "=" * 80)
        print("EXECUTION TRACE")
        print("=" * 80)
        print(f"Query: {query.get('query', 'N/A')}")
        print(f"Timestamp: {query.get('timestamp', 'N/A')}")
        print(f"Total Latency: {query.get('total_latency', 0):.3f}s")
        print(f"Estimated Cost: ${query.get('estimated_cost', 0):.4f}")
        print(f"Steps: {query.get('steps', 0)}")
        print(f"Citations: {query.get('citation_count', 0)}")

        if "tool_calls" in query:
            print(f"\nTool Calls ({len(query['tool_calls'])})")

            for i, tool_call in enumerate(query["tool_calls"], 1):
                print(
                    f"  {i}. {tool_call['name']} "
                    f"({tool_call.get('args', {})})"
                )
                print(
                    f"     → Latency: "
                    f"{tool_call.get('latency', 0):.3f}s"
                )
                print(
                    f"     → Status: "
                    f"{'Success' if tool_call.get('success', True) else '❌ Failed'}"
                )

        print("=" * 80)

In [22]:
metrics = AgentMetrics()

Safety Guardrails

In [30]:
class SafetyGuardrails:
    def __init__(self, max_steps: int = 10, max_cost: float = 1.0):
        self.max_steps = max_steps
        self.max_cost = max_cost
        self.current_cost = 0.0

        self.pii_patterns = [
            (r'\b\d{3}-\d{2}-\d{3,4}\b', '[SSN]'),
            (r'\b\d{12,16}\b', '[CARD]'),
            (
                r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b',
                '[EMAIL]'
            ),
        ]

    def check_step_limit(self, current_step: int) -> bool:
        """Prevent infinite loops."""
        if current_step >= self.max_steps:
            raise ValueError(
                f"Safety: Step limit reached ({self.max_steps}). "
                "Possible infinite loop."
            )
        return True

    def check_cost_limit(self, additional_cost: float) -> bool:
        """Prevent cost explosions."""
        self.current_cost += additional_cost

        if self.current_cost >= self.max_cost:
            raise ValueError(
                f"Safety: Cost limit reached (${self.max_cost}). "
                f"Current: ${self.current_cost}"
            )
        return True

    def mask_pii(self, text: str) -> str:
        """Mask personally identifiable information."""
        masked = text

        for pattern, replacement in self.pii_patterns:
            masked = re.sub(pattern, replacement, masked)

        return masked


    def validate_citations(self, response: str, sources: List[str]) -> bool:
        """Ensure all factual claims have citations."""

        # Count citations in response
        citation_count = len(re.findall(r'\[Source: [^\]]+\]', response))

        # Heuristic: Should have at least one citation per source used
        if citation_count < len(set(sources)):
            print(
                f"Warning: Low citation coverage "
                f"({citation_count} citations for {len(set(sources))} sources)"
            )
            return False

        return True


    def reset_cost(self):
        """Reset cost counter (call per query)."""
        self.current_cost = 0.0

In [38]:
guardrails = SafetyGuardrails(max_steps = 10, max_cost = 1.0)

In [ ]:
#test_text = "My SSN is 123-45-678 and email is john@example.com, card 456723468999"

In [ ]:
#masked = guardrails.mask_pii(test_text)
#print(masked)

My SSN is [SSN] and email is [EMAIL], card [CARD]


In [ ]:
#response_with_citations = "Order shipped on Jan 15 [Source: order_api]. Refund policy [Source: policy_kb]."

#response_without_citations = "Order shipped on Jan 15. Refund takes 7 days."

#print(
    #f"With citations: {guardrails.validate_citations(response_without_citations, ['order_api', 'policy_kb'])}"
#)

With citations: False


connectinf with aegntic_rag

In [37]:
from agentic_rag import build_agent, AgentState, ORDERS_DB

In [52]:
def to_text(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                parts.append(block.get("text", ""))
            else:
                parts.append(str(block))
        return "".join(parts)
    return str(content)

def production_agent_query(query: str) -> Dict[str, Any]:
    """
    Production-ready agent with caching, metrics, and safety.
    """
    start_time = time.time()  # act like a stopwatch
    guardrails.reset_cost()

    # Step 1: Check cache
    cached_response = cache.get(query, estimated_cost=0.05)
    if cached_response:
        end_time = time.time()
        return {
            "query": query,
            "response": cached_response,
            "from_cache": True,
            "latency": end_time - start_time,
            "cost": 0.0
        }

    # Step 2: Execute my Agent

    agent = build_agent()
    initial_state = AgentState(
        query=query,
        messages=[HumanMessage(content=query)],
        tools_results=[],
        sources=[],
        steps=0,
        max_steps=10
    )

    # running the agent
    tool_calls_log = []

    # doing all the safety checks
    try:
        result = agent.invoke(initial_state)
        result["final_response"] = to_text(result["final_response"])
        guardrails.check_step_limit(result["steps"])
        guardrails.validate_citations(result["final_response"], result["sources"])

        safe_response = guardrails.mask_pii(result["final_response"])

        end_time = time.time()
        total_latency = end_time - start_time
        estimated_cost = 0.001 * result["steps"] + 0.01

        citation_count = len(re.findall(r'\[Source: [^\]]+\]', result["final_response"]))

        query_data = {
            "query": query,
            "steps": result["steps"],
            "total_latency": total_latency,
            "estimated_cost": estimated_cost,
            "citation_count": citation_count,
            "tool_calls": [
                {
                    "name": tr.get("source", "unknown"),
                    "args": {},
                    "success": "error" not in tr,
                    "latency": 0.1  # Mock latency
                }
                for tr in result.get("tools_results", [])
            ]
        }

        metrics.log_query(query_data)

        cache.set(query, safe_response)
        print("Final Response: ",safe_response)

        print(f"   Latency: {total_latency:.3f}s")
        print(f"   Est. Cost: ${estimated_cost:.4f}")
        print(f"   Steps: {result['steps']}")
        print(f"   Tools Used: {len(result['tools_results'])}")
        print(f"   Citations: {citation_count}")

        return {
            "query": query,
            "response": safe_response,
            "from_cache": False,
            "latency": total_latency,
            "cost": estimated_cost,
            "steps": result["steps"],
            "citations": citation_count
        }
    except Exception as e:
        print(f"\nError: {e}")
        return {
            "query": query,
            "error": str(e),
            "from_cache": False
        }



In [47]:
result1 = production_agent_query("Where is my order #12345?")

HIT - EXACT


In [42]:
from pprint import pprint

In [44]:
pprint(result1["response"])

'Your order is in transit...'


In [53]:
result2 = production_agent_query("I want to return order #12345 because its damaged.How much refund?")

MISS
Final Response:  Based on the provided evidence, the refund amount for order #12345 is **$119.97**. 

Here are the details from the calculation:
* **Original Amount:** $119.97
* **Restocking Fee:** $0.00
* **Total Refund Amount:** $119.97
* **Reason:** Damaged

**Source:** `refund_calculator` (Source: unknown)
   Latency: 10.410s
   Est. Cost: $0.0120
   Steps: 2
   Tools Used: 1
   Citations: 0
